# 01 — US equity bars → Delta bronze / silver / gold features

**Purpose:** Prepare **completed 5-minute bars** (not incomplete/live ticks) into market-wide, point-in-time-safe features for the ranking model. Feed input table data from your own Kafka/streaming ingestion. This notebook does not itself connect to a broker or market data API.

**Sequence:** Run `01 → 02 → 03` for the demo. In a real 5-minute workflow: ingest/aggregate data upstream, run `01` with `write_mode=incremental`, then `03`. Retrain `02` periodically.

**Required input schema** (UTC `event_time` is the *end* of a completed 5-minute bar):

| Column | Type | Description |
|---|---|---|
| `event_time` | timestamp | UTC, bar-end (e.g. 09:35 ET for 09:30–09:35 bar) |
| `symbol` | string | Stable instrument identifier; do not reuse ticker symbols across delistings |
| `open`, `high`, `low`, `close` | double | Split-adjusted consistently for historical research |
| `volume` | double | Shares in the bar, consistent with split handling |
| `sector` | string | Point-in-time sector; unknown allowed |
| `bid`, `ask`, `bid_size`, `ask_size` | double | Optional synchronized Level-1 snapshot at bar close (not true trade-level OFI) |

**Assumptions:** Equities from a consistent US exchange calendar, regular trading hours, clean unique `(symbol,event_time)` keys. The demo uses artificial prices and is **not** evidence of an investable alpha. Set catalog/schema permissions before running. The `main` Unity Catalog catalog is only a default.


In [ ]:
from pyspark.sql import functions as F, Window as W
from pyspark.sql import types as T
from delta.tables import DeltaTable
from datetime import datetime, timedelta, timezone
import numpy as np, pandas as pd

spark.conf.set("spark.sql.session.timeZone", "UTC")

for name, default in [
    ("data_mode", "demo"),            # demo | table
    ("source_table", "main.market_data.us_equity_5m"),
    ("catalog", "main"),
    ("schema", "stock_alpha"),
    ("write_mode", "bootstrap"),       # bootstrap | incremental
    ("lookback_days", "55"),          # incremental history: at least ~20 US trading days
    ("refresh_days", "2"),
    ("min_price", "5.0"),
    ("min_adv", "500000"),
]:
    dbutils.widgets.text(name, default)

cfg={k:dbutils.widgets.get(k).strip() for k in ["data_mode", "source_table", "catalog", "schema", "write_mode", "lookback_days", "refresh_days", "min_price", "min_adv"]}
assert cfg["data_mode"] in {"demo","table"}
assert cfg["write_mode"] in {"bootstrap","incremental"}
assert cfg["catalog"].replace('_','').isalnum() and cfg["schema"].replace('_','').isalnum()
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{cfg['catalog']}`.`{cfg['schema']}`")
BASE=f"`{cfg['catalog']}`.`{cfg['schema']}`"
BRONZE=f"{BASE}.`equity_bronze`"
SILVER=f"{BASE}.`equity_silver_5m`"
GOLD=f"{BASE}.`equity_features_5m`"
print(f"Data mode={cfg['data_mode']}; write={cfg['write_mode']}; target={BASE}")

## Input: deterministic synthetic universe or your live-data Delta table
The demo generates 48 equities, 46 business days, and 78 completed bars per day. It deliberately uses **synthetic** data with plausible OHLCV shape. Your production ingestion can be Kafka/Kinesis → Structured Streaming → a completed-bar Delta table. Keep this notebook's inputs closed-bar only.

In [ ]:
def make_demo():
    rng=np.random.default_rng(2026)
    days=pd.bdate_range("2026-07-06",periods=46)
    times=[]
    for d in days:
        start=pd.Timestamp(d.date()).tz_localize("America/New_York") + pd.Timedelta(hours=9,minutes=35)
        times.extend([(start+pd.Timedelta(minutes=5*i)).tz_convert("UTC").tz_localize(None) for i in range(78)])
    n=len(times)
    market=rng.normal(0.00001,0.00065,n)
    groups=["Technology","Finance","Health","Industrial","Consumer","Energy"]
    sector_shocks={x:rng.normal(0,0.0004,n) for x in groups}
    frames=[]
    for j in range(48):
        sector=groups[j%len(groups)]
        innovations=rng.normal(0,0.0011,n)
        rets=np.empty(n)
        for i in range(n):
            rets[i]=0.65*market[i]+0.38*sector_shocks[sector][i]+innovations[i]+(0.13*rets[i-1] if i else 0)
        prev=15.0+2.0*j
        closes=prev*np.exp(np.cumsum(rets))
        opens=np.r_[prev,closes[:-1]]
        wiggle=np.abs(rng.normal(0,0.00065,n))
        highs=np.maximum(opens,closes)*(1+wiggle)
        lows=np.minimum(opens,closes)*(1-wiggle)
        minute_index=np.tile(np.arange(78),len(days))
        shape=1.6+1.5*np.abs(minute_index-38.5)/38.5
        volumes=np.maximum(100, rng.lognormal(np.log(10000),0.50,n)*shape).round()
        frames.append(pd.DataFrame({"event_time":times,"symbol":f"DEMO{j:03d}","open":opens,"high":highs,"low":lows,"close":closes,"volume":volumes,"sector":sector}))
    raw=pd.concat(frames,ignore_index=True)
    return spark.createDataFrame(raw)

if cfg["data_mode"] == "demo":
    assert cfg["write_mode"] == "bootstrap", "Demo runs bootstrap only."
    raw=make_demo()
else:
    raw=spark.table(cfg["source_table"])
    if cfg["write_mode"] == "incremental":
        start=datetime.now(timezone.utc)-timedelta(days=int(cfg["lookback_days"]))
        raw=raw.filter(F.col("event_time")>=F.lit(start))

required={"event_time","symbol","open","high","low","close","volume"}
missing=required-set(raw.columns)
if missing: raise ValueError(f"Input missing required columns: {sorted(missing)}")
if "sector" not in raw.columns:
    raw=raw.withColumn("sector",F.lit("Unknown"))
for col in ("bid","ask","bid_size","ask_size"):
    if col not in raw.columns: raw=raw.withColumn(col,F.lit(None).cast("double"))
raw=raw.select(F.col("event_time").cast("timestamp"),F.upper(F.trim("symbol")).alias("symbol"),
               *[F.col(c).cast("double").alias(c) for c in ("open","high","low","close","volume")],
               F.coalesce("sector",F.lit("Unknown")).alias("sector"),
               *[F.col(c).cast("double").alias(c) for c in ("bid","ask","bid_size","ask_size")])
print("Raw rows",raw.count())

## Validate bars, assign New York sessions, calculate past-only liquidity
Input is assumed to be already adjusted for splits. **Never** make historical features using a future-delivered earnings event or a current-day full-session volume. ADV uses **previous** trading sessions only. Duplicate keys are rejected rather than arbitrarily selecting a conflicting price record.

In [ ]:
bad=raw.filter(
    F.col("event_time").isNull()|F.col("symbol").isNull()|(F.length("symbol")==0)|
    (F.col("open")<=0)|(F.col("high")<=0)|(F.col("low")<=0)|(F.col("close")<=0)|
    (F.col("volume")<0)|(F.col("high")<F.greatest("open","close","low"))|
    (F.col("low")>F.least("open","close","high"))
)
if bad.limit(1).count():
    print("Example invalid OHLCV rows:")
    display(bad.limit(10))
    raise ValueError("Invalid OHLCV encountered; clean upstream before features")
if raw.groupBy("symbol","event_time").count().filter("count>1").limit(1).count():
    raise ValueError("Conflicting duplicate symbol/event_time keys: deduplicate upstream with sequence/version")

local_time=F.from_utc_timestamp("event_time","America/New_York")
silver=(raw.withColumn("session_date",F.to_date(local_time))
           .withColumn("minute_of_day",F.hour(local_time)*60+F.minute(local_time))
           .filter((F.col("minute_of_day")>=575)&(F.col("minute_of_day")<=960))
          )  # 09:35 through 16:00 ET, with DST handled by Spark

# ADV from all bars on each day. The lagged window excludes current day entirely.
daily=(silver.groupBy("symbol","session_date")
       .agg(F.sum("volume").alias("total_day_volume")))
dw=W.partitionBy("symbol").orderBy("session_date").rowsBetween(-20,-1)
daily=daily.withColumn("adv_20",F.avg("total_day_volume").over(dw))           .withColumn("adv_days",F.count("total_day_volume").over(dw))
silver=silver.join(daily.select("symbol","session_date","adv_20","adv_days"),["symbol","session_date"],"left")


## Build multi-asset market features
All model inputs come from data known at the **current closed bar** or earlier. Rolling window rows reflect successive observed 5-minute bars (session boundaries may be crossed for momentum/beta); future-target calculations happen only in Notebook 02.

In [ ]:
sw=W.partitionBy("symbol").orderBy("event_time")
w12=sw.rowsBetween(-11,0)
w14=sw.rowsBetween(-13,0)
w60=sw.rowsBetween(-59,0)
slot=W.partitionBy("symbol","minute_of_day").orderBy("event_time").rowsBetween(-20,-1)

features=(silver
    .withColumn("prev_close",F.lag("close",1).over(sw))
    .withColumn("ret_1",F.col("close")/F.col("prev_close")-1)
    .withColumn("ret_3",F.col("close")/F.lag("close",3).over(sw)-1)
    .withColumn("ret_12",F.col("close")/F.lag("close",12).over(sw)-1)
    .withColumn("ret_vol_12",F.stddev_samp("ret_1").over(w12))
    .withColumn("gain",F.greatest(F.col("close")-F.col("prev_close"),F.lit(0.0)))
    .withColumn("loss",F.greatest(F.col("prev_close")-F.col("close"),F.lit(0.0)))
    .withColumn("avg_gain",F.avg("gain").over(w14))
    .withColumn("avg_loss",F.avg("loss").over(w14))
    .withColumn("rsi_14",F.when(F.col("avg_loss")==0,100.0).otherwise(100.0-100.0/(1+F.col("avg_gain")/F.col("avg_loss"))))
    .withColumn("typical_slot_volume",F.avg("volume").over(slot))
    .withColumn("rel_volume",F.when(F.col("typical_slot_volume")>0,F.col("volume")/F.col("typical_slot_volume")))
    .withColumn("clv",F.when(F.col("high")>F.col("low"), (2*F.col("close")-F.col("high")-F.col("low"))/(F.col("high")-F.col("low"))).otherwise(0.0))
    .withColumn("amihud",F.abs("ret_1")/F.greatest(F.col("close")*F.col("volume"),F.lit(1.0)))
    .withColumn("book_imbalance",F.when((F.col("bid_size")+F.col("ask_size"))>0,
               (F.col("bid_size")-F.col("ask_size"))/(F.col("bid_size")+F.col("ask_size"))))
    .withColumn("spread_bps",F.when((F.col("bid")>0)&(F.col("ask")>=F.col("bid")),
               1e4*(F.col("ask")-F.col("bid"))/((F.col("ask")+F.col("bid"))/2)))
)

market=(features.groupBy("event_time")
        .agg(F.avg("ret_1").alias("market_ret")))
mw=W.orderBy("event_time").rowsBetween(-11,0)
market=(market.withColumn("market_vol_12",F.stddev_samp("market_ret").over(mw))
       .withColumn("market_mom_12",F.sum("market_ret").over(mw)))
features=features.join(market,"event_time","left")
features=(features
    .withColumn("rolling_cov",F.covar_samp("ret_1","market_ret").over(w60))
    .withColumn("rolling_market_var",F.var_samp("market_ret").over(w60))
    .withColumn("beta_60",F.when(F.col("rolling_market_var")>1e-12,F.col("rolling_cov")/F.col("rolling_market_var")))
    .withColumn("residual_mom_12",F.col("ret_12")-F.col("beta_60")*F.col("market_mom_12")))

# Eligibility is based on completed-bar price and lagged ADV; preserve source history for rolling windows.
eligible=features.filter((F.col("close")>=float(cfg["min_price"])) &
    (F.col("adv_20")>=float(cfg["min_adv"]))&(F.col("adv_days")>=15))
sector_sort=W.partitionBy("event_time","sector").orderBy("ret_3")
sector_vol_sort=W.partitionBy("event_time","sector").orderBy("rel_volume")
market_sort=W.partitionBy("event_time").orderBy("residual_mom_12")
features_gold=(eligible
    .withColumn("sector_ret_rank",F.percent_rank().over(sector_sort))
    .withColumn("sector_rvol_rank",F.percent_rank().over(sector_vol_sort))
    .withColumn("market_resid_rank",F.percent_rank().over(market_sort))
    .select("event_time","symbol","sector","session_date","minute_of_day","close","volume","adv_20",
        "ret_1","ret_3","ret_12","ret_vol_12","rsi_14","rel_volume","clv","amihud","beta_60",
        "residual_mom_12","book_imbalance","spread_bps","sector_ret_rank","sector_rvol_rank",
        "market_resid_rank","market_ret","market_vol_12","market_mom_12"))
print("Eligible rows",features_gold.count())

In [ ]:
# Invariants worth checking before writing any training data.
if features_gold.groupBy('event_time','symbol').count().filter('count>1').limit(1).count():
    raise ValueError('Gold features have duplicate primary keys')
if features_gold.filter((F.col('sector_ret_rank')<0)|(F.col('sector_ret_rank')>1)|
                        (F.col('market_resid_rank')<0)|(F.col('market_resid_rank')>1)).limit(1).count():
    raise ValueError('Invalid cross-sectional percentile rank')
print('Feature keys and rank-range checks passed')

## Persist Delta tables
`bootstrap` creates full history. `incremental` reads a historical lookback, recomputes features, and **MERGEs only recent days** to keep prior rows. Choose a lookback long enough for 20 previous regular sessions and keep all symbols in a timestamp together. Late changes before `refresh_days` require explicit backfill. The optional Level-1 fields are bar-aligned snapshots, **not** true order-flow imbalance.

In [ ]:
def persist(df,table):
    if cfg["write_mode"]=="bootstrap" or not spark.catalog.tableExists(table.replace('`','')):
        (df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table))
        return
    min_refresh=datetime.now(timezone.utc)-timedelta(days=int(cfg["refresh_days"]))
    rows=df.filter(F.col("event_time")>=F.lit(min_refresh))
    (DeltaTable.forName(spark,table).alias("t")
       .merge(rows.alias("s"),"t.symbol=s.symbol AND t.event_time=s.event_time")
       .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

persist(raw,BRONZE)
persist(silver,SILVER)
persist(features_gold,GOLD)
print("Bronze:",BRONZE,"Silver:",SILVER,"Gold:",GOLD)
display(spark.table(GOLD).orderBy(F.col("event_time").desc(),"symbol").limit(20))

**Operational notes:** For true 5-minute ingestion, stream upstream into 5-minute closed bars with watermark/late-data policy and a durable checkpoint. Schedule this notebook only after the complete 5-minute bar arrives for the entire universe, then run Notebook 03. Incremental `MERGE` of Gold does not delete rows that become invalid after an upstream correction; handle corrections with a separate reconciliation job. Model retraining uses full historical Silver and Gold (Notebook 02).